# DocFill Agent — Clinical Study Report Demo

This notebook shows how an LLM agent can automatically fill a DOCX template using:

1. **DOCX → AST**: parse the template into a structured element tree (headings, paragraphs, table cells)
2. **MCP tools**: expose upload / inspect / edit / validate as callable tools
3. **Agno agent**: the LLM reads the AST, understands the structure, fills blanks from meeting notes
4. **Run-level formatting preservation**: bold labels in the template stay bold; LLM values stay plain

**Requirements**
```
pip install -e .[dev]
# Start the MCP server in a separate terminal:
# uvicorn docfill.mcp.server:app --port 8000
```

Set your API key in `.env` (copy from `.env.example`).

## 0. Setup

In [1]:
import os
from pathlib import Path

from dotenv import load_dotenv
load_dotenv()

# Generate the CSR template if it doesn't exist yet
template_path = Path("templates/csr_template.docx")
if not template_path.exists():
    import subprocess, sys
    subprocess.run([sys.executable, "create_template.py"], check=True)
    print("Template created.")
else:
    print(f"Template found: {template_path}")

Template found: templates/csr_template.docx


## 1. Inspect the raw AST (no agent)

Let's see what the parser sees before involving any LLM.

In [2]:
import sys
sys.path.insert(0, "../../src")

from docscribe.ast.converter import build_ast

ast = build_ast(template_path)
print(f"Total elements: {ast.total_elements}")
print()

for el in ast.elements:
    kind = el.kind
    eid  = el.element_id
    text = getattr(el, "text", "")[:60]
    runs = getattr(el, "runs", [])
    bold_runs = [r.text[:20] for r in runs if r.bold]
    print(f"[{kind:12s}] {eid:30s}  text={repr(text):62s}  bold_labels={bold_runs}")

Total elements: 124

[paragraph   ] para-0                          text='CLINICAL STUDY REPORT'                                         bold_labels=['CLINICAL STUDY REPOR']
[paragraph   ] para-1                          text='[FICTIONAL COMPOUND — FOR DEMONSTRATION ONLY]'                 bold_labels=[]
[paragraph   ] para-2                          text=''                                                              bold_labels=[]
[heading     ] heading-3                       text='1. Study Identification'                                       bold_labels=[]
[table_cell  ] cell-t0-r0-c0                   text='Study Title:'                                                  bold_labels=['Study Title:']
[table_cell  ] cell-t0-r0-c1                   text=''                                                              bold_labels=[]
[table_cell  ] cell-t0-r1-c0                   text='Protocol Number:'                                              bold_labels=['Protocol Number:']
[table_c

## 2. Sample clinical notes (input to the agent)

> These are **fictional** notes for a fictional compound. They represent what a study team might paste into a chat interface.

In [3]:
STUDY_NOTES = """
Study: A Phase III randomised, double-blind, placebo-controlled trial
evaluating the efficacy and safety of Zetaribumab 150 mg SC Q4W in adults
with active generalised myasthenia gravis (gMG) with anti-AChR antibodies.

Protocol: ZMG-301
EudraCT: 2024-001234-56
Phase: III
Sponsor: Fictional Pharma Ltd.
Coordinating Investigator: Dr. Elena Vasquez, MD PhD
Start: 01 March 2022 | End: 28 February 2024 | Report: 15 July 2024

INN: Zetaribumab | Proprietary: Zeta-GM | ATC: L04AA99
Formulation: 150 mg/mL solution for injection | Route: Subcutaneous
Dose: 150 mg Q4W | Manufacturer: BioFictional SA | Lot: ZMG-2022-A

Primary endpoint: Change from baseline in QMG total score at Week 26
  Treatment: n=110, Mean change = -9.2 ± 4.1, p < 0.001
  Control:   n=108, Mean change = -3.8 ± 3.9

Secondary endpoints:
  SE1: MG-ADL responder rate (≥3 pt improvement) — Treatment 68% vs Control 32%, p<0.001
  SE2: MGC total score change at Week 26 — Treatment -7.1 vs Control -2.9, p=0.003
  SE3: Neuro-QoL Fatigue T-score change — Treatment -8.4 vs Control -3.1, p=0.009

Safety (n=218 exposed):
  AEs: Treatment 78%, Control 74%
  SAEs: Treatment 12%, Control 14%
  Discontinuations: Treatment 6%, Control 5%
  Deaths: 0 (both arms)
  Most common AEs (≥5%):
    Nasopharyngitis: T 14% / C 12%
    Headache:        T 11% / C 13%
    Injection-site reaction: T 18% / C 4%
    Upper respiratory tract infection: T 9% / C 8%
    Fatigue: T 7% / C 9%

Efficacy conclusion: Zetaribumab met its primary endpoint with a statistically
significant and clinically meaningful reduction in QMG score vs placebo.

Safety conclusion: The safety profile was consistent with the known class effects;
no new safety signals were identified.

Benefit-risk: The benefit-risk profile of Zetaribumab 150 mg Q4W is favourable
in anti-AChR-positive gMG patients. Study team: Dr. A. Fontaine (Medical Officer),
J. Chen PhD (Statistician), Dr. P. Nakamura (Clinical Pharmacologist).
"""
print("Notes ready — length:", len(STUDY_NOTES), "chars")

Notes ready — length: 1950 chars


## 3. Build the agent

The agent connects to the local MCP server (`http://localhost:8000`) and receives
the same 5-step system prompt that drives the workflow:
`upload → inspect → load_ast → edit → validate`.

In [4]:
from agno.agent import Agent
from agno.models.litellm import LiteLLMOpenAI
from agno.tools.mcp import MCPTools

MCP_URL      = "http://localhost:8000/mcp"   # FastMCP 3.x endpoint
MODEL_ID     = os.getenv("MODEL_ID", "bedrock-claude-4-sonnet")
LITELLM_HOST = os.getenv("LITELLM_HOST", "http://0.0.0.0:4000")
LITELLM_KEY  = os.getenv("LITELLM_API_KEY", "")

SYSTEM_PROMPT = """\
You are DocFill, an expert document assistant.
Your job is to fill an empty DOCX template with information extracted from user notes.

WORKFLOW — execute all steps automatically:

STEP 1  Call get_session_documents(). If the template is already uploaded, skip to STEP 3.

STEP 2  Call upload_document(file_path='examples/csr_demo/templates/csr_template.docx',
        file_id='csr-template').

STEP 3  Call load_document_ast(file_id='csr-template').
        Read EVERY element carefully. Note the EXACT element_id of each empty table cell.

STEP 4  Call edit_document(file_id='csr-template', edits=[...]).
        Rules:
        - Use element_id EXACTLY as returned by load_document_ast. Never invent IDs.
        - Only fill cells that are empty or contain a placeholder like '[...]'.
        - Do NOT invent data. If a value is not in the notes, leave the cell empty.

STEP 5  Call validate_document_state(file_id='csr-template', create_new_version=True).

Execute all steps without asking for confirmation.
"""

print(f"Config — model: {MODEL_ID}, proxy: {LITELLM_HOST}, MCP: {MCP_URL}")

Config — model: bedrock-claude-4-sonnet, proxy: https://alxn-genio-llm-dev.aig.astrazeneca.net/, MCP: http://localhost:8000/mcp


In [5]:
## Diagnostic + run — MCPTools initialized inside async context manager

async def run():
    async with MCPTools(url=MCP_URL, transport="streamable-http") as mcp_tools:
        # Show tools fetched from server
        fns = getattr(mcp_tools, "functions", {})
        print(f"MCP tools loaded: {list(fns.keys()) if fns else '(none yet)'}")

        agent = Agent(
            name="DocFill",
            model=LiteLLMOpenAI(id=MODEL_ID, api_key=LITELLM_KEY, base_url=LITELLM_HOST),
            tools=[mcp_tools],
            instructions=SYSTEM_PROMPT,
            markdown=True,
        )

        fns_after = getattr(mcp_tools, "functions", {})
        print(f"Tools on agent: {list(fns_after.keys())}\n")

        user_msg = f"""Fill the CSR template using these study notes.
Template path: examples/csr_demo/templates/csr_template.docx

STUDY NOTES:
{STUDY_NOTES}
"""
        response = await agent.arun(user_msg)
        return response

response = await run()
print(response.content)

MCP tools loaded: ['upload_document', 'get_session_documents', 'load_document_ast', 'edit_document', 'validate_document_state']
Tools on agent: ['upload_document', 'get_session_documents', 'load_document_ast', 'edit_document', 'validate_document_state']

I'll help you fill the CSR template with the study information provided. Let me start by checking if the template is already uploaded and then proceed with the workflow.The template is not uploaded yet. Let me upload it first.Now let me load the document AST to see the structure and identify which fields need to be filled.Perfect! Now I can see the template structure. I'll fill in all the empty fields with the information from the study notes. Let me create the edits:I see there's an issue with the table numbering. Let me re-examine the AST more carefully and use the correct table indices. Looking at the AST, I need to identify the actual table indices that exist in the document.I see the issue - I was mistakenly trying to edit the lab

## 5. Inspect result

After the agent runs, the filled DOCX is at `docfill_store/csr-template/output/`.
Let's verify it via the AST.

In [ ]:
backend = os.getenv("STORE_BACKEND", "local").lower()

if backend == "s3":
    from docscribe.storage.s3 import S3DocumentStore
    result_store = S3DocumentStore()
    print(f"Inspecting S3 store: s3://{result_store.bucket}/{result_store.folder}/")
else:
    from docscribe.storage.local import LocalDocumentStore
    result_store = LocalDocumentStore("../../docscribe_store")
    print(f"Inspecting local store: {result_store.root.resolve()}")

filled_path = result_store.get_file("csr-template", is_output=True)

if filled_path:
    print(f"Filled DOCX: {filled_path}")
    filled_ast = build_ast(filled_path)
    print(f"Total elements: {filled_ast.total_elements}\n")
    for el in filled_ast.elements:
        text = getattr(el, "text", "")[:80]
        if text.strip():
            print(f"[{el.kind:12s}] {el.element_id:30s}  {repr(text)}")
else:
    print("Output file not found — did the agent complete successfully?")

## 6. Run formatting preservation check

This cell verifies that bold labels were preserved and values are plain.

In [8]:
if filled_path:
    ok = 0
    issues = []
    for el in filled_ast.elements:
        if el.kind != "table_cell" or not el.text.strip():
            continue
        for run in (el.runs or []):
            # Labels (ending in ':') should be bold; values should be plain
            stripped = run.text.strip()
            if stripped.endswith(":") and not run.bold:
                issues.append(f"{el.element_id}: label '{stripped}' is not bold")
            elif not stripped.endswith(":") and stripped and run.bold:
                # Values shouldn't be bold (heuristic — may have false positives)
                pass  # lenient check
        ok += 1
    if issues:
        for issue in issues:
            print("WARN:", issue)
    else:
        print(f"Formatting check passed on {ok} non-empty table cells.")

Formatting check passed on 68 non-empty table cells.
